# Lab 04 (Databricks Edition): Jira Agent Test Generation

Generate Credit Risk Assessment test cases from a Jira story with Databricks Model Serving

**What you'll build:** a notebook that pulls a Jira story, sends it to a model on Databricks, and saves the test cases it writes.

**Flow:** Widgets + Jira API token → Jira REST API → Requirement extractor → LLM (Databricks Model Serving) → Markdown parser → Delta table + CSV/JSON in a Unity Catalog Volume.

**Runtime:** Databricks Runtime 15.4 LTS or above. You don't need `%pip install`, a GPU, or a model download.

## Before you run
1. Create a Jira API token from your Atlassian account (https://id.atlassian.com/manage-profile/security/api-tokens).
2. Paste it into the `JIRA_API_TOKEN` variable in the Configuration cell. It's a plain Python variable here to keep the lab simple, not a widget.
3. Don't commit this notebook or share it with the token inside.

## 0. Setup
Your instructor has already set up the storage this lab needs, so you don't have to create anything. Run the notebook from the top.

If you see any error, ask your instructor for help.

## 1. Configuration

In [0]:
# Replace the placeholders with your Jira details and the issue key you want to test.
dbutils.widgets.text("jira_base_url", "https://udaysomapuram.atlassian.net")
dbutils.widgets.text("jira_email", "udaysomapuram@gmail.com")
dbutils.widgets.text("issue_key", "MDP-10")
dbutils.widgets.text("endpoint_name", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.text("catalog", "credit_lab")
dbutils.widgets.text("schema", "lab")
dbutils.widgets.text("volume", "output")

JIRA_BASE_URL = dbutils.widgets.get("jira_base_url").strip().rstrip("/")
JIRA_EMAIL    = dbutils.widgets.get("jira_email").strip()
ISSUE_KEY     = dbutils.widgets.get("issue_key").strip()
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name").strip()
CATALOG       = dbutils.widgets.get("catalog").strip()
SCHEMA_NAME   = dbutils.widgets.get("schema").strip()
VOLUME        = dbutils.widgets.get("volume").strip()

# Paste your Jira API token between the quotes. Don't share or commit this notebook.
JIRA_API_TOKEN = "ATATT3xFfGF0YULvTdRQFthFLLLK0YCh-MsLpXLJE7cRfFO4D4PLrjxMVBbE5v7Wpau8SM3wuSA1YxFaVKl4sNXnvC_lLZpSjbOg4nV9cYjep4rVErWAMzRPBf534ZwgdPnISEM_EDqlE9F468lTP6jHRCtKbm1hNmzpp3-dTTO74kHU1A849RM=6402F147"

assert JIRA_BASE_URL and JIRA_EMAIL and ISSUE_KEY, "Fill in the jira_base_url, jira_email and issue_key widgets."
assert JIRA_API_TOKEN and "PASTE_YOUR" not in JIRA_API_TOKEN, "Paste your Jira API token into JIRA_API_TOKEN."
print(f"Config OK. Issue: {ISSUE_KEY} | Endpoint: {ENDPOINT_NAME}")

Quick check that your storage is ready before we call Jira or the model.

In [0]:
try:
    spark.sql(f"DESCRIBE VOLUME {CATALOG}.{SCHEMA_NAME}.{VOLUME}")
    print("Storage is ready.")
except Exception as e:
    raise RuntimeError("Storage isn't set up for you yet. Please ask your instructor for help.") from e

## 2. Jira connector

In [0]:
import json
from typing import Any, Dict
import requests

def fetch_jira_issue(issue_key: str) -> Dict[str, Any]:
    """Fetch a Jira issue by key via the Jira Cloud v3 REST API."""
    if not issue_key:
        raise ValueError("Set the issue_key widget before fetching an issue.")
    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{issue_key}"
    response = requests.get(url, auth=(JIRA_EMAIL, JIRA_API_TOKEN), timeout=30)
    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        # Don't print the response body. It can contain sensitive data.
        print(f"Jira API error: {error} (HTTP {response.status_code})")
        raise
    return response.json()

# Smoke test
test_issue = fetch_jira_issue(ISSUE_KEY)
print(f"Fetched {ISSUE_KEY}. Type: {test_issue['fields']['issuetype']['name']}")

## 3. Requirement extractor
Jira Cloud stores rich-text descriptions in Atlassian Document Format (ADF), which is nested JSON. Sending that raw JSON to the model wastes tokens and adds noise, so `adf_to_text` flattens it into plain text first.

In [0]:
def adf_to_text(node: Any) -> str:
    """Flatten a Jira ADF node (or list of nodes) into plain text."""
    if node is None:
        return ""
    if isinstance(node, str):
        return node
    if isinstance(node, list):
        return "\n".join(t for t in (adf_to_text(n) for n in node) if t)

    node_type = node.get("type")
    children = node.get("content", [])

    if node_type == "text":
        return node.get("text", "")
    if node_type == "hardBreak":
        return "\n"
    if node_type == "mention":
        return node.get("attrs", {}).get("text", "")
    if node_type == "inlineCard":
        return node.get("attrs", {}).get("url", "")
    if node_type in ("paragraph", "codeBlock"):
        return "".join(adf_to_text(c) for c in children)
    if node_type == "heading":
        level = node.get("attrs", {}).get("level", 1)
        return "#" * level + " " + "".join(adf_to_text(c) for c in children)
    if node_type == "listItem":
        return "- " + adf_to_text(children).replace("\n", "\n  ")
    if node_type == "tableRow":
        return " | ".join(adf_to_text(c).replace("\n", " ") for c in children)
    return adf_to_text(children)   # doc, bulletList, orderedList, table, panel, and so on

def normalize_field(field_value: Any) -> str:
    """Turn a Jira field into prompt-friendly text."""
    if field_value is None:
        return ""
    if isinstance(field_value, str):
        return field_value
    text = adf_to_text(field_value).strip()
    # If flattening finds nothing, fall back to the raw JSON so nothing is lost.
    return text or json.dumps(field_value, indent=2, ensure_ascii=False)

def extract_requirements(issue_json: Dict[str, Any]) -> Dict[str, str]:
    """Extract the story summary and description."""
    fields = issue_json.get("fields", {})
    return {
        "summary": normalize_field(fields.get("summary", "")),
        "description": normalize_field(fields.get("description", "")),
    }

reqs = extract_requirements(test_issue)
print("Summary:", reqs["summary"])
print("Description (first 500 characters):\n", reqs["description"][:500])

## 4. LLM setup: Databricks Model Serving
Databricks Model Serving gives you access to the model you pick through a serving endpoint. The SDK signs you in automatically inside the notebook, so there's nothing to download and no extra key to manage.

The first cell below lists the `databricks-` endpoints in your workspace. The default, `databricks-meta-llama-3-3-70b-instruct`, only works if it appears in that list. If it doesn't, pick another chat model from the list and change the `endpoint_name` widget.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

# Endpoints available in your workspace
for e in w.serving_endpoints.list():
    if e.name.startswith("databricks-"):
        print(e.name)

def generate_with_llm(system_prompt: str, user_prompt: str, max_tokens: int = 6000) -> str:
    """Call a Databricks serving endpoint and return the text of the reply."""
    resp = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
            ChatMessage(role=ChatMessageRole.USER, content=user_prompt),
        ],
        max_tokens=max_tokens,
        temperature=0.0,   # same input, same output, as the story's non-functional requirements ask
    )
    return resp.choices[0].message.content.strip()

## 5. Prompt
Two things to notice in this prompt. It gives a count range wide enough for everything the story asks for, and it defines what Positive, Negative and Edge mean, so the model labels cases by the scenario and not by whether the business outcome sounds good or bad.

In [0]:
SYSTEM_PROMPT = "You are a senior QA engineer. You output only Markdown tables, exactly as instructed."

BASE_TESTGEN_PROMPT = """
Read the Jira story requirements below and generate between 20 and 30 test cases covering:
- Company Registration Certificate and GST Certificate validation
- Company-name matching and MCA status
- Sanctions screening and analyst review
- Two years of financial statements
- Current Ratio, Debt-to-Equity Ratio, and Net Profit Margin calculations
- D&B score and Industry Risk Rating
- Final score, risk category, credit limit, and payment terms
- Auditability, response time, data protection, and validation errors
- Positive, negative, edge, and boundary scenarios

Use the scorecard weights and thresholds exactly as supplied. Do not invent scoring logic.

Include tests for:
- Ratio bands
- D&B scores of 700, 650, and 600
- Every industry matrix category
- Final scores of 49, 50, 80, and 81

IMPORTANT OUTPUT RULES:
- Return ONLY the Markdown table. No explanations, intro text, or conclusions.
- Every data row must contain exactly 8 columns.
- The Type value must be exactly one of: Positive, Negative, Edge.
- Use Positive for valid business scenarios, Negative for invalid, missing, or rejected input, and Edge for boundary values (exactly on a threshold) or unusual conditions.
- Do not use the pipe character (|) inside any cell content. Use commas, semicolons, or words instead.
- Number the test cases TC-001, TC-002, and so on.

Format the answer as a Markdown table with exactly these columns:

| ID | Title | Type (Positive/Negative/Edge) | Pre-conditions | Steps | Expected Result | Mapped Requirement(s) | Priority |

Jira story requirements:

Summary:
{summary}

Description:
{description}
"""

def build_testcase_prompt(requirements: Dict[str, str]) -> str:
    # replace() instead of format(), so curly braces in the Jira text can't break the template
    return (BASE_TESTGEN_PROMPT
            .replace("{summary}", requirements["summary"])
            .replace("{description}", requirements["description"]))

## 6. The test generation agent

In [0]:
class JiraTestGenerationAgent:
    def __init__(self, issue_key: str):
        self.issue_key = issue_key

    def run(self) -> str:
        issue_json = fetch_jira_issue(self.issue_key)
        requirements = extract_requirements(issue_json)
        prompt = build_testcase_prompt(requirements)

        print("Prompt sent to the model (first 1000 characters):")
        print(prompt[:1000], "...\n")

        output = generate_with_llm(SYSTEM_PROMPT, prompt)

        # The prompt asks for at least 20 cases. Far fewer usually means the reply was cut off.
        n_cases = output.count("TC-")
        if n_cases < 20:
            print(f"WARNING: only {n_cases} test cases found. "
                  "The output may be truncated. Try a higher max_tokens.")
        return output

## 7. Run the agent

In [0]:
agent = JiraTestGenerationAgent(issue_key=ISSUE_KEY)
testcases_markdown = agent.run()
print(testcases_markdown)

## 8. Parse the table, then save it
The model returns a Markdown table. This cell turns it into rows, saves a Delta table, and writes a CSV and a JSON file to the volume.

Column names are converted to snake_case because Delta doesn't accept spaces or parentheses in them by default. If a row doesn't have exactly 8 columns, the cell stops and tells you which one. That means the model broke the format, so re-run the agent or try a stronger endpoint.

In [0]:
import csv, io
from datetime import datetime, timezone

# ---------- 1. Parse the Markdown table ----------
raw = testcases_markdown.strip()
if raw.startswith("```markdown"):             # remove code fences if the model added them
    raw = raw[len("```markdown"):].strip()
elif raw.startswith("```"):
    raw = raw[3:].strip()
if raw.endswith("```"):
    raw = raw[:-3].strip()

table_lines = []
for line in raw.splitlines():
    line = line.strip()
    if "|" not in line:
        continue
    cells = [c.strip() for c in line.strip("|").split("|")]
    if all(c.replace("-", "").replace(":", "").strip() == "" for c in cells):
        continue                               # skip the |---|---| separator row
    table_lines.append(line)

assert len(table_lines) >= 2, (
    "No Markdown table found. The model may not have followed the format. "
    "Re-run the agent, or switch to a stronger endpoint."
)

headers = [c.strip() for c in table_lines[0].strip("|").split("|")]
assert len(headers) == 8, f"Expected 8 columns in the header, got {len(headers)}: {headers}"

rows = []
for row_line in table_lines[1:]:
    cells = [c.strip() for c in row_line.strip("|").split("|")]
    assert len(cells) == len(headers), (
        f"Expected {len(headers)} columns, got {len(cells)} in this row: {cells}"
    )
    rows.append(cells)
print(f"Parsed {len(rows)} test cases. Columns: {headers}")

# ---------- 2. Delta-safe column names, plus two audit columns ----------
COLUMNS = ["test_id", "title", "test_type", "preconditions",
           "steps", "expected_result", "mapped_requirements", "priority"]
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
records = []
for cells in rows:
    rec = dict(zip(COLUMNS, cells))
    rec["source_issue"] = ISSUE_KEY
    rec["generated_at"] = now
    records.append(rec)

# ---------- 3. Delta table in Unity Catalog ----------
df = spark.createDataFrame(records)
table_name = f"{CATALOG}.{SCHEMA_NAME}.credit_risk_test_cases"
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
print(f"Delta table written: {table_name}")

# ---------- 4. CSV and JSON in the volume ----------
vol = f"/Volumes/{CATALOG}/{SCHEMA_NAME}/{VOLUME}"
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=COLUMNS, extrasaction="ignore")
writer.writeheader()
writer.writerows(records)
dbutils.fs.put(f"{vol}/{ISSUE_KEY}_credit_risk_testcases.csv", buf.getvalue(), overwrite=True)
dbutils.fs.put(f"{vol}/{ISSUE_KEY}_credit_risk_testcases.json",
               json.dumps(records, ensure_ascii=False, indent=2), overwrite=True)
print(f"CSV and JSON saved under {vol}/")

display(df)   # read through these before you trust them

## 9. Query the results
Edge cases are the boundary tests, such as a score of exactly 50 or a D&B score of exactly 600.

In [0]:
%sql
SELECT * FROM credit_lab.lab.credit_risk_test_cases WHERE test_type = 'Edge'

And a count per type:

In [0]:
%sql
SELECT test_type, COUNT(*) AS total FROM credit_lab.lab.credit_risk_test_cases GROUP BY test_type

## Extensions
- Attach the CSV back to the Jira issue through the REST API.
- Split the work across three agents: one reads requirements, one designs tests, one reviews them (Mosaic AI Agent Framework).
- Pull a dedicated Acceptance Criteria field if your Jira project has one.

## Summary
You built a Databricks workflow that reads a Jira story, asks a served foundation model for test cases, and saves them in a Delta table and in CSV and JSON files in a Unity Catalog Volume.

**Key lesson:** the model writes a first draft, not a finished test suite. Read the cases before they go into your test management process, and check them against the story. For example, did it cover final scores of 80 and 81, and every industry category?